# Quality Sensitivity To Trimming

        ## Manuscript targets
        - `Methodology / Integrated Quality Sensitivity Analysis`
- `Results / Figure Spearman correlation quality plot`

        ## Primary inputs
        - `share/results/technical/per_srr_eval.tsv`
- `share/results/technical/per_srr_quality.tsv`
- `data/flattened_fastqc_raw/ or share/data/flattened_fastqc_raw/`

        ## Rebuild scripts
        - `share/scripts/analysis/01_extract_per_srr_fastqc_metrics.py`
- `share/scripts/analysis/02_evaluate_featurecounts_matrix_similarity.py`

        This notebook prefers the full local `data/` working set when it exists and falls back to the tiny `share/data/` example bundle otherwise.


## Inputs, methodology, and rebuild policy

The manuscript asks whether samples with worse terminal quality are more sensitive to trimming-induced count-profile changes. This notebook merges reconstructed terminal-quality estimates with the count-profile concordance table and visualizes the trend across trimming modes.


In [ ]:
from __future__ import annotations

import math
import re
import subprocess
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="talk")
except ImportError:
    sns = None

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 220)
plt.rcParams["figure.dpi"] = 120
SRR_PATTERN = re.compile(r"(SRR\d+)")


def find_repo_root() -> Path:
    start = Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / "share").exists() and (candidate / "share" / "results").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root from the notebook working directory.")


REPO_ROOT = find_repo_root()
SHARE = REPO_ROOT / "share"
RESULTS = SHARE / "results"
TECH = RESULTS / "technical"
BIO = RESULTS / "biological"
SUPP = RESULTS / "supplementary"
SHARE_DATA = SHARE / "data"
LOCAL_DATA = REPO_ROOT / "data"
SCRIPTS = SHARE / "scripts"
NOTEBOOK_DB = SHARE / "notebooks" / "srr_queue.db"


def data_dir(name: str) -> Path:
    local = LOCAL_DATA / name
    if local.exists():
        return local
    return SHARE_DATA / name


def load_table(path: Path) -> pd.DataFrame:
    sep = "," if path.suffix == ".csv" else "\t"
    return pd.read_csv(path, sep=sep)


def run_script(script_rel: str, *args: str) -> None:
    script = REPO_ROOT / script_rel
    if not script.exists():
        raise FileNotFoundError(script)
    cmd = ["python3", str(script), *map(str, args)] if script.suffix == ".py" else ["bash", str(script), *map(str, args)]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=REPO_ROOT)


def parse_fastqc_terminal_quality(fastqc_data_path: Path) -> float | None:
    rows: list[list[str]] = []
    in_section = False
    for line in fastqc_data_path.read_text(encoding="utf-8", errors="replace").splitlines():
        if line.startswith(">>Per base sequence quality"):
            in_section = True
            continue
        if in_section and line.startswith(">>END_MODULE"):
            break
        if in_section and line and not line.startswith("#"):
            rows.append(line.split("\t"))
    if not rows:
        return None
    means = []
    for row in rows:
        try:
            means.append(float(row[1]))
        except (IndexError, ValueError):
            continue
    if not means:
        return None
    return float(np.mean(means[-min(10, len(means)) :]))


def load_terminal_quality_table() -> pd.DataFrame:
    quality = load_table(TECH / "per_srr_quality.tsv")
    raw_dir = data_dir("flattened_fastqc_raw")
    rows = []
    if raw_dir.exists():
        for fastqc_data in raw_dir.glob("*/*_fastqc/fastqc_data.txt"):
            terminal_q = parse_fastqc_terminal_quality(fastqc_data)
            if terminal_q is None:
                continue
            project_id = fastqc_data.parents[1].name
            srr = fastqc_data.parent.name.removesuffix("_fastqc")
            if srr.endswith("_1") or srr.endswith("_2"):
                srr = srr[:-2]
            rows.append({
                "project_id": project_id,
                "SRR_ID": srr,
                "terminal_q_mean": terminal_q,
            })
    if rows:
        terminal = (
            pd.DataFrame(rows)
            .groupby(["project_id", "SRR_ID"], as_index=False)["terminal_q_mean"]
            .mean()
        )
        merged = quality.merge(terminal, on=["project_id", "SRR_ID"], how="left")
        merged["terminal_q_mean"] = merged["terminal_q_mean"].fillna(
            merged["Q_mean"] - merged["tail_quality_decay"]
        )
        return merged

    quality["terminal_q_mean"] = quality["Q_mean"] - quality["tail_quality_decay"]
    return quality


def srr_ids_from_tree(root: Path) -> set[str]:
    srrs: set[str] = set()
    if not root.exists():
        return srrs
    for path in root.rglob("*"):
        match = SRR_PATTERN.search(path.name) or SRR_PATTERN.search(path.as_posix())
        if match:
            srrs.add(match.group(1))
    return srrs


def flattened_dir_srr_sets(root: Path, cohort_projects: set[str] | None = None) -> dict[str, set[str]]:
    dir_sets: dict[str, set[str]] = {}
    if not root.exists():
        return dir_sets

    for flattened_dir in sorted(
        p for p in root.iterdir() if p.is_dir() and p.name.startswith("flattened_")
    ):
        dir_srrs: set[str] = set()
        for project_dir in sorted(p for p in flattened_dir.iterdir() if p.is_dir()):
            if cohort_projects is not None and project_dir.name not in cohort_projects:
                continue
            dir_srrs.update(srr_ids_from_tree(project_dir))
        dir_sets[flattened_dir.name] = dir_srrs
    return dir_sets


def collect_flattened_dir_audit(root: Path, cohort_projects: set[str] | None = None) -> pd.DataFrame:
    rows = []
    if not root.exists():
        return pd.DataFrame(
            columns=[
                "flattened_dir",
                "projects_total",
                "unique_srrs_total",
                "projects_in_cohort",
                "unique_srrs_in_cohort",
                "projects_outside_cohort",
                "unique_srrs_outside_cohort",
                "outside_cohort_projects",
            ]
        )

    for flattened_dir in sorted(
        p for p in root.iterdir() if p.is_dir() and p.name.startswith("flattened_")
    ):
        project_srrs: dict[str, set[str]] = {}
        for project_dir in sorted(p for p in flattened_dir.iterdir() if p.is_dir()):
            project_srrs[project_dir.name] = srr_ids_from_tree(project_dir)

        all_srrs = set().union(*project_srrs.values()) if project_srrs else set()
        if cohort_projects is None:
            in_cohort = project_srrs
            out_cohort: dict[str, set[str]] = {}
        else:
            in_cohort = {k: v for k, v in project_srrs.items() if k in cohort_projects}
            out_cohort = {k: v for k, v in project_srrs.items() if k not in cohort_projects}

        in_srrs = set().union(*in_cohort.values()) if in_cohort else set()
        out_srrs = set().union(*out_cohort.values()) if out_cohort else set()

        rows.append(
            {
                "flattened_dir": flattened_dir.name,
                "projects_total": len(project_srrs),
                "unique_srrs_total": len(all_srrs),
                "projects_in_cohort": len(in_cohort),
                "unique_srrs_in_cohort": len(in_srrs),
                "projects_outside_cohort": len(out_cohort),
                "unique_srrs_outside_cohort": len(out_srrs),
                "outside_cohort_projects": ", ".join(sorted(out_cohort)) if out_cohort else "",
            }
        )

    return pd.DataFrame(rows)


In [ ]:
quality = load_terminal_quality_table()[["SRR_ID", "project_id", "terminal_q_mean", "Q_mean"]]
eval_df = load_table(TECH / "per_srr_eval.tsv")

long_rows = []
spearman_cols = {
    "Adapter": "untrmd_adptrTrmd_spear",
    "P5": "untrmd_P5Trmd_spear",
    "P10": "untrmd_P10Trmd_spear",
    "P20": "untrmd_P20Trmd_spear",
    "P35": "untrmd_P35Trmd_spear",
}
for method, col in spearman_cols.items():
    tmp = eval_df[["SRR_ID", "project_id", col]].copy()
    tmp = tmp.rename(columns={col: "spearman"})
    tmp["method"] = method
    long_rows.append(tmp)

quality_long = pd.concat(long_rows, ignore_index=True)
quality_long["spearman"] = pd.to_numeric(quality_long["spearman"], errors="coerce")
merged = quality_long.merge(quality, on=["SRR_ID", "project_id"], how="left").dropna()
merged.head()


In [ ]:
bins = np.arange(math.floor(merged["terminal_q_mean"].min()), math.ceil(merged["terminal_q_mean"].max()) + 1, 1.0)
merged["terminal_quality_bin"] = pd.cut(merged["terminal_q_mean"], bins=bins, include_lowest=True)
trend = (
    merged.groupby(["method", "terminal_quality_bin"], observed=False)["spearman"]
    .median()
    .reset_index()
)
trend["terminal_quality_mid"] = trend["terminal_quality_bin"].apply(
    lambda interval: interval.mid if pd.notna(interval) else np.nan
)
display(trend.head(15))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

if sns is not None:
    sns.scatterplot(
        data=merged,
        x="terminal_q_mean",
        y="spearman",
        hue="method",
        alpha=0.25,
        s=20,
        ax=axes[0],
    )
else:
    for method, grp in merged.groupby("method"):
        axes[0].scatter(grp["terminal_q_mean"], grp["spearman"], alpha=0.25, s=20, label=method)
    axes[0].legend()
axes[0].set_title("Per-sample Spearman vs terminal quality")
axes[0].set_xlabel("Terminal mean quality")
axes[0].set_ylabel("Spearman correlation")

for method, grp in trend.groupby("method"):
    axes[1].plot(grp["terminal_quality_mid"], grp["spearman"], marker="o", linewidth=2, label=method)
axes[1].set_title("Median Spearman by terminal-quality bin")
axes[1].set_xlabel("Terminal mean quality bin midpoint")
axes[1].set_ylabel("Median Spearman correlation")
axes[1].legend()

fig.tight_layout()
plt.show()
